# 1. Manually developing a Naïve Bayes model

a.	Generate a Bag of Words (for word frequency)

In [36]:
import pandas as pd
from collections import Counter
import math

# Creating the dataset
data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

df = pd.DataFrame(data, columns=['doc', 'class'])

In [ ]:
from collections import Counter

def tokenize(text):
    # This ensures punctuation is removed so words like "manager." match "manager"
    clean_text = text.lower().replace("!", "").replace("?", "").replace(",", "").replace(".", "")
    return clean_text.split()

# Lists to store all tokens found in each class
spam_words = []
ham_words = []

for _, row in df.iterrows():
    tokens = tokenize(row['doc'])
    if row['class'] == 'SPAM':
        spam_words.extend(tokens)
    else:
        ham_words.extend(tokens)

# Generate frequency counts
spam_counts = Counter(spam_words)
ham_counts = Counter(ham_words)

# Create a set of all unique words (The Vocabulary)
vocabulary = set(spam_words + ham_words)

# --- PRINTING THE VOCABULARY SUMMARY ---
print("--- DATASET STATISTICS ---")
print(f"Total Unique Words (Vocab Size |V|): {len(vocabulary)}")
print(f"Total Tokens in SPAM (N_spam):      {len(spam_words)}")
print(f"Total Tokens in HAM (N_ham):        {len(ham_words)}")
print("-" * 30)
print(f"Unique words in SPAM: {len(spam_counts)}")
print(f"Unique words in HAM:  {len(ham_counts)}")
print("-" * 30)
print(f"Full Vocabulary:\n{sorted(list(vocabulary))}")

--- DATASET STATISTICS ---
Total Unique Words (Vocab Size |V|): 44
Total Tokens in SPAM (N_spam):      22
Total Tokens in HAM (N_ham):        33
------------------------------
Unique words in SPAM: 20
Unique words in HAM:  25
------------------------------
Full Vocabulary:
['3', '50%', 'a', 'are', 'at', 'can', 'catch', 'click', 'dinner', 'for', 'free', 'get', 'here', 'hi', 'how', 'in', 'iphone', "let's", 'limited', 'lowest', 'meds', 'meeting', 'mom', 'money', 'now', 'off', 'office', 'on', 'pm', 'price', 'prizes', 'report', 'send', 'still', 'team', 'the', 'time', 'today', 'tomorrow', 'up', 'we', 'win', 'you', 'your']


b.	Calculate the prior for the class HAM and SPAM

In [38]:
total_docs = len(df)
p_spam = len(df[df['class'] == 'SPAM']) / total_docs
p_ham = len(df[df['class'] == 'HAM']) / total_docs

print(f"Prior SPAM: {p_spam:.2f}")
print(f"Prior HAM: {p_ham:.2f}")

Prior SPAM: 0.45
Prior HAM: 0.55


In [39]:
def get_likelihood(word, class_counts, total_words_in_class, vocab_size):
    return (class_counts[word] + 1) / (total_words_in_class + vocab_size)

# Totals for denominator
total_spam_tokens = len(spam_words)
total_ham_tokens = len(ham_words)
vocab_size = len(vocabulary)

c.	Calculate the likelihood of the tokens in the vocabulary with respect to the class.

d.	Determine the class of the following test sentence:

In [48]:
def predict_and_print(sentence):
    tokens = tokenize(sentence)
    
    # Starting values (Priors) - using raw probability
    spam_score = p_spam
    ham_score = p_ham
    
    print(f"--- RESULTS FOR: '{sentence}' ---")
    print(f"{'Word':<15} | {'P(w|SPAM)':<15} | {'P(w|HAM)':<15}")
    print("-" * 55)

    for word in tokens:
        # Get raw likelihoods
        prob_s = get_likelihood(word, spam_counts, total_spam_tokens, vocab_size)
        prob_h = get_likelihood(word, ham_counts, total_ham_tokens, vocab_size)
        
        # Multiply the scores
        spam_score *= prob_s
        ham_score *= prob_h
        
        print(f"{word:<15} | {prob_s:<15.4f} | {prob_h:<15.4f}")

    print("-" * 55)
    
    # Determining result
    final_class = "SPAM" if spam_score > ham_score else "HAM"

    # Using scientific notation (:.2e) because these numbers are tiny!
    summary = (
        f"TOTAL PROBABILITY: SPAM = {spam_score:.2e} | HAM = {ham_score:.2e}\n"
        f"RESULTS:           {final_class}\n"
        f"{'='*60}\n"
    )
    print(summary)

# Run them
predict_and_print("Limited offer, click here!")
predict_and_print("Meeting at 2 PM with the manager.")

--- RESULTS FOR: 'Limited offer, click here!' ---
Word            | P(w|SPAM)       | P(w|HAM)       
-------------------------------------------------------
limited         | 0.0303          | 0.0130         
offer           | 0.0152          | 0.0130         
click           | 0.0303          | 0.0130         
here            | 0.0303          | 0.0130         
-------------------------------------------------------
TOTAL PROBABILITY: SPAM = 1.92e-07 | HAM = 1.55e-08
RESULTS:           SPAM

--- RESULTS FOR: 'Meeting at 2 PM with the manager.' ---
Word            | P(w|SPAM)       | P(w|HAM)       
-------------------------------------------------------
meeting         | 0.0152          | 0.0390         
at              | 0.0152          | 0.0390         
2               | 0.0152          | 0.0130         
pm              | 0.0152          | 0.0260         
with            | 0.0152          | 0.0130         
the             | 0.0152          | 0.0519         
manager         | 0.0152

# 2.	Using Scikit-Learn. 

In [43]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# 1. Prepare and Train
docs = [
    "Free money now!!!", "Hi mom, how are you?", "Lowest price for your meds",
    "Are we still on for dinner?", "Win a free iPhone today",
    "Let's catch up tomorrow at the office", "Meeting at 3 PM tomorrow",
    "Get 50% off, limited time!", "Team meeting in the office",
    "Click here for prizes!", "Can you send the report?"
]
labels = ["SPAM", "HAM", "SPAM", "HAM", "SPAM", "HAM", "HAM", "SPAM", "HAM", "SPAM", "HAM"]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(docs)

model = MultinomialNB()
model.fit(X_train, labels)

# --- NEW: Extracting Priors and Likelihoods ---

# A. Print Priors (class_log_prior_ stores log probabilities)
priors = np.exp(model.class_log_prior_)
print("--- CLASS PRIORS (Scikit-Learn) ---")
for i, label in enumerate(model.classes_):
    print(f"P({label}): {priors[i]:.4f}")

# B. Print Likelihoods for each word in the vocabulary
# feature_log_prob_ stores log(P(word|class))
words = vectorizer.get_feature_names_out()
likelihoods = np.exp(model.feature_log_prob_)

print("\n--- TOKEN LIKELIHOODS (Top 10 samples) ---")
print(f"{'Word':<15} | {'P(w|HAM)':<12} | {'P(w|SPAM)':<12}")
print("-" * 45)
for i in range(len(words)):
    # Showing just a selection to keep it clean
    if words[i] in ['free', 'money', 'meeting', 'click', 'tomorrow']:
        print(f"{words[i]:<15} | {likelihoods[0][i]:.4f}     | {likelihoods[1][i]:.4f}")

# --- 5. Predict and show result ---
test_sentences = ["Limited offer, click here!", "Meeting at 2 PM with the manager."]
X_test = vectorizer.transform(test_sentences)
predictions = model.predict(X_test)
probs = model.predict_proba(X_test) # Probability for each class

print("\n--- TEST PREDICTIONS ---")
for i, sentence in enumerate(test_sentences):
    print(f"Sentence: '{sentence}'")
    print(f"Result: {predictions[i]} (Confidence: {np.max(probs[i]):.4f})")
    print("-" * 30)

--- CLASS PRIORS (Scikit-Learn) ---
P(HAM): 0.5455
P(SPAM): 0.4545

--- TOKEN LIKELIHOODS (Top 10 samples) ---
Word            | P(w|HAM)     | P(w|SPAM)   
---------------------------------------------
click           | 0.0135     | 0.0317
free            | 0.0135     | 0.0476
meeting         | 0.0405     | 0.0159
money           | 0.0135     | 0.0317
tomorrow        | 0.0405     | 0.0159

--- TEST PREDICTIONS ---
Sentence: 'Limited offer, click here!'
Result: SPAM (Confidence: 0.9153)
------------------------------
Sentence: 'Meeting at 2 PM with the manager.'
Result: HAM (Confidence: 0.9784)
------------------------------
